Pricing Life Insurances using Australian Mortality Data

Recently I have been studying for various actuarial exams, and in order to improve my coding and to apply my newfound knowledge, I worked on this project, which explores life insurance pricing. Using real mortality data, we construct an actuarial pricing framework to compute/estimate premiums for different kinds of life insurance contracts and under different assumptions.

All calculations are implemented in Python.



1. Import libraries and load the Australian mortality data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the excel file with the Australian life table sheets
xls = pd.ExcelFile("data/Australian_Life_Tables.xlsx")

# Extract the "ALT 2020-22 Males" excel sheet and change the header by 1 (otherwise the columns are unnamed)
df = pd.read_excel(
    "data/Australian_Life_Tables.xlsx",
    sheet_name="ALT 2020-22 Males",
    header = 1
)

# For now we only need to have the age and the qx columns

df = df[["Age", "qx"]]

# Print a small piece of the table
df.head()

# Check below !!!
#df.to_csv("data/aus_mortality.csv", index=False)
#df = pd.read_csv("data/aus_mortality.csv")


In [ ]:

# Create a dictionary using the table above
mortality = dict(zip(df["Age"], df["qx"]))

plt.figure(figsize=(10,6))

plt.plot(df["Age"], df["qx"])

plt.yscale("log")

plt.xlabel("Age")
plt.ylabel("Mortality Rate qx")

plt.title("Australian Mortality Rates")

plt.grid(True)

plt.show()


We see a relatively high mortality among newborns, then a sharp decrease, and at the age of 30 it seems to take on a more exponential growth, till around the age of 90, where it seems the death rates are slowing. From what I have found online, the "late-life mortality plateau" is still somewhat debated. In any case, let's try to fit a Gompertz law of mortality among the ages 30-90.

To this end, we will assume q_x = A*exp(Bx) and to make the fit better (though this might not be necessary), consider log(q_x) = log(A) +Bx. We use linear regression to find the best fit for A and B. 

In [ ]:
df_fit = df[
    (df["Age"] >= 30)
    & (df["Age"] <= 80)
]
ages = df_fit["Age"].values
qx = df_fit["qx"].values

# Get rid of 0 values (if they exist)
mask = qx > 0
ages = ages[mask]
qx = qx[mask]

# Take the log
log_qx = np.log(qx)

# logA is the y-intercept and B is the slope of the linear fit
B_fit, logA_fit = np.polyfit(
    ages,
    log_qx,
    1
)

A_fit = np.exp(logA_fit)

print("A =", A_fit)
print("B =", B_fit)

# The fitted mortality rate
qx_fit = A_fit * np.exp(B_fit * ages)

plt.figure(figsize=(10,6))

plt.plot(ages, qx, label="Observed")

plt.plot(
    ages,
    qx_fit,
    '--',
    label="Gompertz Fit"
)


plt.yscale("log")

plt.xlabel("Age")
plt.ylabel("Mortality Rate qx")

plt.title("Observed vs Gompertz Mortality Fit")

plt.legend()
plt.grid(True)

plt.show()


The Gompertz model seems to capture the overall exponential increase in mortality with age (at least, between the ages 30 and 90), but the fitted curve tends to slightly overestimate observed mortality rates across much of the age range considered (till age 80). This suggests that, although the Gompertz law provides a useful approximation for adult mortality, real mortality data exhibits features that are not fully captured by this model.



Life Insurance Premium Calculation

Our next plan is to now calculate premiums for a term life insurance contract, using both the fitted data and the observed data. We consider a term insurance with a fixed benefit payable at the end of the year of death, monthly premium payments, and expenses.

The premium is determined using the equivalence principle:

EPV(Premiums) = EPV(Benefits) + EPV(Expenses)


For now we will consider the following assumptions:

age = 40 (exact)

term = 20

benefit = 100 000

interest = 4%

initial expense = 200  + 50% of 1st premium

renewal expense = 5% of one monthly premium

We use a constant force of mortality 

Note that the equivalence principle then gives us:
Premium  = ( EPV(benefits) + initial expense_fixed )  /  (  annuity_factor - initial expense_percentage - renewal expense * annuity_factor  )

Our next plan is to calculate the premium for a life insurance and to plot a graph showing the premium per age.

In [ ]:

term = 20

sum_assured = 100000

interest = 0.04

# Monthly premiums
payments_per_year = 12

# Expense assumptions
initial_expense_fixed = 200

initial_expense_percent = 0.50

renewal_expense_percent = 0.05

# Define the relevant discount factors

v = 1 / (1 + interest)

i_month = (1 + interest)**(1/12) - 1

v_month = 1 / (1 + i_month)


# Death rate using constant force of mortality
def monthly_q(q_annual):
    return 1 - (1 - q_annual)**(1/12)

# Create the list of premiums for the ages 20-80
premium_list = []

ages_test = range(20,81)

for age in ages_test:

    #Compute the expected present value of the benefits
    epv_benefits = 0
    survival = 1

    for k in range(term):

        current_age = age + k

        q = mortality[current_age]

        death_prob = survival * q

        # The benefit is paid at the end of the year i.e. the benefir would be sum_assured * v^{k+1}
        epv_benefits += (
        sum_assured
        * (v ** (k + 1))
        * death_prob
        )

        survival *= (1 - q)
    
    #Compute the epv of the annuity (used to compute the epv of the premiums paid)
    annuity = 0
    survival = 1

    for t in range(term * 12):

        current_age = age + t // 12
        q_m = monthly_q(mortality[current_age])

        annuity += (
        (v_month ** t)
        * survival
        )

        survival *= (1 - q_m)
    
    P = (
    epv_benefits + initial_expense_fixed
    ) / (
    annuity
    - initial_expense_percent
    - renewal_expense_percent * annuity
    )
    premium_list.append(P)



plt.figure(figsize=(10,6))

plt.plot(ages_test, premium_list)

plt.xlabel("Age")

plt.ylabel("Monthly Premium")

plt.title("Monthly Premium by Entry Age")

plt.grid(True)

plt.show()

Note that the premiums increase quickly with age (especially after the age of 60). This is in line with the Gompertz model of mortality, which models mortality as being exponential, and hence the EPV of the benefits goes up, while the EPV of the premium annuity goes down (and hence the montly premium goes up). 

We will now model how the premiums vary if the interest rate varies. We assume the exact same assumptions as before, but fix the age at 45 and let the interest change. 

In [ ]:
age = 45

term = 20

sum_assured = 100000

interest = 0.04

# Monthly premiums
payments_per_year = 12

# Expense assumptions
initial_expense_fixed = 200

initial_expense_percent = 0.50

renewal_expense_percent = 0.05



# Death rate using constant force of mortality
def monthly_q(q_annual):
    return 1 - (1 - q_annual)**(1/12)

# Create the list of premiums for the ages 20-80
premium_list = []

interest_rates = np.arange(0.01, 0.101, 0.005)

for interest in interest_rates:

    # Define the relevant discount factors

    v = 1 / (1 + interest)

    i_month = (1 + interest)**(1/12) - 1

    v_month = 1 / (1 + i_month)

    #Compute the expected present value of the benefits
    epv_benefits = 0
    survival = 1
    for k in range(term):

        current_age = age + k

        q = mortality[current_age]

        death_prob = survival * q

        # The benefit is paid at the end of the year i.e. the benefir would be sum_assured * v^{k+1}
        epv_benefits += (
        sum_assured
        * (v ** (k + 1))
        * death_prob
        )

        survival *= (1 - q)
    
    #Compute the epv of the annuity (used to compute the epv of the premiums paid)
    annuity = 0
    survival = 1

    for t in range(term * 12):

        current_age = age + t // 12
        q_m = monthly_q(mortality[current_age])

        annuity += (
        (v_month ** t)
        * survival
        )

        survival *= (1 - q_m)
    
    P = (
    epv_benefits + initial_expense_fixed
    ) / (
    annuity
    - initial_expense_percent
    - renewal_expense_percent * annuity
    )
    premium_list.append(P)



plt.figure(figsize=(10,6))

plt.plot(interest_rates, premium_list)

plt.xlabel("Interest")

plt.ylabel("Monthly Premium")

plt.title("Monthly Premium by Interest")

plt.grid(True)

plt.show()





We note that the premiums decrease as the interest rate increases because future claim payments are discounted more heavily. At lower interest rates, we require higher premiums since future investment earnings are reduced.
